# Week 1 — LLM Foundations

**Ships today:** `hr-assistant` v0 — a streaming terminal chatbot with a live cost meter.

**The four ideas.** Everything in the next nine weeks rests on these:

1. The model only predicts the next token.
2. Tokens are the unit of money *and* the unit of space.
3. **The model has no memory.** Every call is fresh.
4. There are two kinds of model, and temperature only applies to one of them.

**How to run this.** Top to bottom. Later cells use objects defined earlier.

**One habit to build today.** After every call, ask *"what did the model actually
receive?"* — and be able to answer it. The `show()` helper below exists only for that,
and it will still be in use in week 9 when we are debugging a voice agent.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

print("Environment configured successfully")

Environment configured successfully


## LLM Via API call 
LLMs exposes the gateway via HTTP like any other typical microservices
- Chatgpt vs gpt
- Models are stateless
- models has no memory

In [ ]:
import httpx
def call_llm(prompt):
    url = "https://api.openai.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "gpt-4o-mini",
        "messages": [{"role": "user", "content": prompt}],
    }
    http_client = httpx.Client()
    response = http_client.post(url, headers=headers, json=data)
    response.raise_for_status()
    return response.json()

call_llm("Hello, how are you?")
    

---

## Part 0 — Setup

Two cells of plumbing: a client, and the two models we compare all session.

In [2]:
# Dependencies (already in the venv):  pip install openai tiktoken
import os
import time
import json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("client ready")

client ready


### The two models

Week 1 uses exactly two models, and the difference between them is a lesson in itself.

> **Before delivery:** verify both model ids and all four prices against the provider's
> current pricing page. Model names and prices move; everything downstream reads these
> constants, so this cell is the only place to edit.

In [3]:
SIMPLE_MODEL   = "gpt-4o-mini"   # a sampling model: predicts, no private reasoning
THINKING_MODEL = "o4-mini"       # a reasoning model: thinks privately, then answers

# USD per 1M tokens. PLACEHOLDERS — update from the provider's pricing page.
PRICING = {
    SIMPLE_MODEL:   {"in": 0.15, "out": 0.60},
    THINKING_MODEL: {"in": 1.10, "out": 4.40},
}

def cost_of(model, in_tokens, out_tokens):
    """Dollar cost of one call."""
    p = PRICING[model]
    return (in_tokens * p["in"] + out_tokens * p["out"]) / 1_000_000

---

## Part 1 — What is a token?

Before any code that counts them, be clear on what is being counted.

**A model cannot read text. It only does arithmetic on numbers.** So before anything
happens, your text is chopped into pieces, and each piece is looked up in a fixed
vocabulary to get an id — a number. Those pieces are **tokens**.

A token is not a letter and not a word. It is a chunk that the vocabulary happened to
learn during training:

- Common English words are usually **one** token — `leave`, `policy`, `the`
- Rare or technical words shatter into **several** — `unenforceability`
- The leading space is normally part of the *following* token — `" leave"`, not `"leave"`
- Numbers, emoji, code and non-English text are often **several tokens each**
- Rough rule for English prose: **1 token ≈ 4 characters ≈ ¾ of a word**

The vocabulary was fixed when the model was trained. You cannot add to it, and you do not
get to choose how your text is split.

### How a request actually becomes a response

```mermaid
flowchart LR
    A["REQUEST TEXT<br/><i>How many leave days<br/>do I get?</i>"]
    --> B["TOKENIZATION<br/>text → token IDs"]

    B --> C["MODEL<br/>predict probability<br/>distribution for next token"]

    C --> D["TOKEN SELECTION<br/>e.g. 'You'"]

    D -->|"append token<br/>and predict next token"| C

    D -->|"stop condition<br/>reached"| E["DETOKENIZATION<br/>token IDs → text"]

    E --> F["RESPONSE TEXT<br/><i>You get 18 days.</i>"]
```

**Read the loop arrow twice — it is the whole mechanism.** The model does not write a
sentence. It predicts *one* token, that token is appended to the input, and the whole
thing runs again. A 300-token answer means **300 passes through the model.**

Three consequences you will feel all term, all of them visible in that diagram:

| Because | Therefore |
|---|---|
| Output is generated one token at a time | Long answers are slow, and you can **stream** them as they appear (Part 10) |
| Each pass re-reads the entire input | Output tokens cost more than input tokens |
| The model only ever sees the token ids you sent it | **It has no memory of any earlier call** (Part 6) |

### See it for yourself

Do not take the split on trust. Look at it.

In [7]:
import os
import pathlib
import tiktoken

ENCODING = "o200k_base"   # the vocabulary used by current OpenAI models

def load_encoding(name=ENCODING):
    """Load the tokenizer normally; fall back to the repo's cache only if that fails.

    tiktoken downloads its vocabulary on first use. On networks that inspect TLS traffic
    that download dies with 'certificate verify failed', so the vocabulary is also
    committed to this repo under tiktoken_cache/ as a backstop.
    """
    try:
        return tiktoken.get_encoding(name), "downloaded (or already in tiktoken's own cache)"
    except Exception as first_error:
        for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
            if (d / "tiktoken_cache").is_dir():
                os.environ["TIKTOKEN_CACHE_DIR"] = str(d / "tiktoken_cache")
                try:
                    return tiktoken.get_encoding(name), f"offline, from {d.name}/tiktoken_cache"
                except Exception:
                    break
        raise RuntimeError(
            f"Could not load the tokenizer: {type(first_error).__name__}: {first_error}\n"
            "No usable tiktoken_cache/ either. Fix it once, from the repo root:\n"
            "    ./.venv/bin/python tools/fetch_tokenizer_cache.py"
        ) from first_error

_enc, _source = load_encoding()
print(f"tokenizer ready — {_enc.n_vocab:,} tokens in the vocabulary  [{_source}]\n")

sample = "Employees are entitled to 18 days of earned leave per calendar year."
ids = _enc.encode(sample)

print(f"{len(sample)} characters  ->  {len(ids)} tokens\n")
for i, tid in enumerate(ids):
    print(f"  {i:>2}  id={tid:<8} {_enc.decode([tid])!r}")

tokenizer ready — 200,019 tokens in the vocabulary  [offline, from learn-ai/tiktoken_cache]

68 characters  ->  14 tokens

   0  id=62497    'Employees'
   1  id=553      ' are'
   2  id=31941    ' entitled'
   3  id=316      ' to'
   4  id=220      ' '
   5  id=1157     '18'
   6  id=3376     ' days'
   7  id=328      ' of'
   8  id=22639    ' earned'
   9  id=7668     ' leave'
  10  id=777      ' per'
  11  id=18712    ' calendar'
  12  id=1284     ' year'
  13  id=13       '.'


Look for the leading spaces inside the quotes. Now break it deliberately:

In [8]:
for text in ["leave", " leave", "Leave", "unenforceability", "Mahadev", "18", "1,80,000", "नमस्ते", "🎉"]:
    ids = _enc.encode(text)
    print(f"{text!r:<20} {len(ids)} token(s)   {[_enc.decode([i]) for i in ids]}")

'leave'              1 token(s)   ['leave']
' leave'             1 token(s)   [' leave']
'Leave'              1 token(s)   ['Leave']
'unenforceability'   3 token(s)   ['unen', 'force', 'ability']
'Mahadev'            2 token(s)   ['Mah', 'adev']
'18'                 1 token(s)   ['18']
'1,80,000'           5 token(s)   ['1', ',', '80', ',', '000']
'नमस्ते'             4 token(s)   ['न', 'म', 'स्त', 'े']
'🎉'                  2 token(s)   ['�', '�']


`"leave"` and `" leave"` are **different tokens**. A name may cost four tokens. An emoji
can cost more than a word. This is why "roughly 4 characters" is a rule of thumb and never
a substitute for counting.

### Now the helper

`count_tokens()` is just `len()` of that list. It matters because from here on, every token
count you see — in `show()`, in the cost meter, in week 3's budget — comes from this one
function.

In [9]:
def count_tokens(text):
    return len(_enc.encode(text))

def tokens_in(messages):
    """Close enough for teaching. Exact per-message overhead varies by model."""
    return sum(count_tokens(m["content"]) + 4 for m in messages)

print(count_tokens("How many days of leave do I get?"), "tokens")

9 tokens


---

## Part 2 — The measurement harness

Three helpers carry all ten weeks: `show()`, `llm()`, and `table()`. Learn them once, here.

They exist because of one rule: **never guess what the model received, or what it cost.**
Both are printable, so print them.

### `show()` — the most important function in this course

`show()` prints the **exact** message array going over the wire, with a token count per
message. Every time output surprises you, run `show()` before you touch the prompt.

More than half the bugs in the next nine weeks are answered by looking at this output and
noticing that something you assumed was in the context simply is not.

In [10]:
def show(messages, title="CONTEXT SENT TO THE MODEL"):
    total = tokens_in(messages)
    print("=" * 72)
    print(f"{title} — {len(messages)} messages, ~{total} tokens")
    print("=" * 72)

    for i, m in enumerate(messages):
        body = m["content"]
        print(f"[{i}] {m['role']} ~{count_tokens(body)} tok | {body}")

    print("=" * 72)
    return total

### `llm()` — one call, fully measured

Every call in this notebook goes through `llm()`. It returns the text and records what the
call cost, so the running total at the end of the session is real rather than guessed.

In [11]:
SPEND = []  # every call appends one record

def llm(messages, model=SIMPLE_MODEL, **kwargs):
    started = time.time()
    resp = client.chat.completions.create(model=model, messages=messages, **kwargs)
    elapsed = time.time() - started

    u = resp.usage
    # Reasoning models spend tokens thinking that you pay for but never see.
    details = getattr(u, "completion_tokens_details", None)
    reasoning = getattr(details, "reasoning_tokens", 0) or 0

    SPEND.append({
        "model": model,
        "in": u.prompt_tokens,
        "out": u.completion_tokens,
        "reasoning": reasoning,
        "seconds": elapsed,
        "usd": cost_of(model, u.prompt_tokens, u.completion_tokens),
    })
    return resp.choices[0].message.content


def spend_summary():
    if not SPEND:
        return "no calls yet"
    total = sum(r["usd"] for r in SPEND)
    hidden = sum(r["reasoning"] for r in SPEND)
    line = f"{len(SPEND)} calls   ${total:.6f}"
    if hidden:
        line += f"   ({hidden} reasoning tokens you paid for and never saw)"
    return line

### `table()` — so comparison cells show the comparison, not the formatting

In [26]:
from tabulate import tabulate
def table(data, headers):
    # print(f"tabulate {data}")
    res = tabulate(data, headers=headers, tablefmt="grid")
    print(res)

---

## Part 3 — The model only predicts the next token

You have seen the pipeline. Now look inside the box marked **MODEL**.

It does not "decide what to say". For a given input it produces a **probability
distribution over the entire vocabulary** — a score for every possible next token — and
then one is picked. That is the whole operation.

We can print that distribution. Ask for one token and request the top candidates.

In [27]:
resp = client.chat.completions.create(
    model=SIMPLE_MODEL,
    messages=[{"role": "user", "content": "Complete this sentence with one word: The capital of France is"}],
    max_tokens=1,
    logprobs=True,
    top_logprobs=5,
)

import math
top = resp.choices[0].logprobs.content[0].top_logprobs
table(
    [[repr(t.token), f"{math.exp(t.logprob):.4%}"] for t in top],
    ["candidate next token", "probability"],
)

+------------------------+---------------+
| candidate next token   | probability   |
+========================+===============+
| 'Paris'                | 99.9997%      |
+------------------------+---------------+
| 'Berlin'               | 0.0001%       |
+------------------------+---------------+
| 'par'                  | 0.0001%       |
+------------------------+---------------+
| ' Paris'               | 0.0001%       |
+------------------------+---------------+
| 'France'               | 0.0000%       |
+------------------------+---------------+


**That table is the model.** Not a metaphor for it — that is literally what it computes.
Everything downstream — chat, RAG, agents, the voice helpline in week 9 — is that table,
sampled from, appended, and computed again.

Try a genuinely ambiguous prompt and watch the probability mass spread out:

In [29]:
resp = client.chat.completions.create(
    model=SIMPLE_MODEL,
    messages=[{"role": "user", "content": "Write one word describing Monday:"}],
    max_tokens=1, logprobs=True, top_logprobs=5,
)
top = resp.choices[0].logprobs.content[0].top_logprobs
table([[repr(t.token), f"{math.exp(t.logprob):.1%}"] for t in top], ["candidate", "probability"])

+-------------+---------------+
| candidate   | probability   |
+=============+===============+
| 'Fresh'     | 66.6%         |
+-------------+---------------+
| 'Renew'     | 8.0%          |
+-------------+---------------+
| 'New'       | 7.0%          |
+-------------+---------------+
| 'Mot'       | 6.2%          |
+-------------+---------------+
| 'Beg'       | 1.2%          |
+-------------+---------------+


Confident question, one candidate near 100%. Open question, the mass spreads.

**Hold this picture.** In Part 7 we change `temperature`, and all it does is reshape this
distribution before a token is picked — flatten it and you get variety, sharpen it and you
get the safe answer. Knowing that makes temperature obvious rather than magic.

---

## Part 4 — Tokens are money and tokens are space

Two consequences, both of which will shape every design decision in this course:

- **Money:** you pay per token, in and out, on every single call.
- **Space:** the context window is a fixed token budget, and your input and the model's
  output share it.

In [30]:
policy = """LEAVE POLICY (extract)
Employees are entitled to 18 days of earned leave per calendar year, accrued monthly.
A maximum of 30 days may be carried forward. Leave must be applied for at least 3
working days in advance, except in the case of illness or emergency. Unused leave
beyond the carry-forward cap lapses on 31 December."""

t = count_tokens(policy)
rows = [
    ["this policy extract", t, f"${cost_of(SIMPLE_MODEL, t, 0):.6f}"],
    ["40 such documents",  t * 40, f"${cost_of(SIMPLE_MODEL, t * 40, 0):.6f}"],
    ["...on every call, 100 calls/day", t * 40 * 100, f"${cost_of(SIMPLE_MODEL, t * 40 * 100, 0):.4f}"],
]
table(rows, ["what you send", "input tokens", "cost"])

+---------------------------------+----------------+-----------+
| what you send                   |   input tokens | cost      |
+=================================+================+===========+
| this policy extract             |             75 | $0.000011 |
+---------------------------------+----------------+-----------+
| 40 such documents               |           3000 | $0.000450 |
+---------------------------------+----------------+-----------+
| ...on every call, 100 calls/day |         300000 | $0.0450   |
+---------------------------------+----------------+-----------+


Hold on to that last row. In **week 3** we will try to put all 40 HR policies into the
context and watch it break — and this is the bill that forces us into RAG in week 5.

---

## Part 5 — You send a list of messages, not a string

Three roles, and each one means something different to the model:

| Role | What it is | HR analogy |
|---|---|---|
| `system` | Standing rules, always in force | Company policy |
| `user` | What the person just asked | The employee's question |
| `assistant` | What the model said earlier | Your own previous replies |

```mermaid
flowchart LR
    S["system<br/>the rules"] --> API["messages = [ ... ]"]
    U["user<br/>the question"] --> API
    A["assistant<br/>earlier replies"] --> API
    API --> M(("model"))
    M --> R["one reply"]
```

In [31]:
messages = [
    {"role": "system", "content": "You are an HR assistant at a software company. Answer in at most 3 sentences."},
    {"role": "user",   "content": "How many days of earned leave do I get?"},
]

show(messages)
reply = llm(messages)
print("\nREPLY:", reply)
print("\n", spend_summary())

CONTEXT SENT TO THE MODEL — 2 messages, ~36 tokens
[0] system ~18 tok | You are an HR assistant at a software company. Answer in at most 3 sentences.
[1] user ~10 tok | How many days of earned leave do I get?

REPLY: You typically receive 15 days of earned leave per year, which may be accrued monthly. Additionally, unused leave may roll over depending on company policy. For specific details, please refer to the employee handbook or consult with HR.

 1 calls   $0.000033


Notice the model answered confidently without ever seeing the leave policy. It is drawing
on training data, not on your company's rules.

**This is the single most important failure mode in the course**, and the entire reason
weeks 3 through 6 exist. Hold the thought — Part 6 makes it worse.

---

## Part 6 — The model has NO memory

Do not take this on trust. Prove it.

Every API call is completely fresh. The model does not remember your last call, your last
question, or that you exist. Whatever it needs to know, **you send again, every time.**

### The experiment: tell it something, then ask about it in a separate call

In [32]:
# Call 1 — give it a fact.
first = llm([
    {"role": "system", "content": "You are an HR assistant."},
    {"role": "user",   "content": "My employee ID is E-4417 and I work in the Pune office."},
])
print("CALL 1:", first)

# Call 2 — a brand new call. Ask about the fact.
second = llm([
    {"role": "system", "content": "You are an HR assistant."},
    {"role": "user",   "content": "What is my employee ID?"},
])
print("\nCALL 2:", second)

CALL 1: Hello! How can I assist you today with your employee ID E-4417 and your inquiries related to the Pune office?

CALL 2: I'm sorry, but I don't have access to personal data, including employee IDs. You may need to check your employee records, company directory, or reach out to your HR department for assistance.


Call 2 has no idea. Not because the model is weak — because **you did not send the
information.** From the model's side, call 2 is the first thing that has ever happened.

### The fix, and the entire basis of every chatbot you have ever used

Resend the conversation. All of it. Every turn.

In [33]:
with_history = [
    {"role": "system",    "content": "You are an HR assistant."},
    {"role": "user",      "content": "My employee ID is E-4417 and I work in the Pune office."},
    {"role": "assistant", "content": first},
    {"role": "user",      "content": "What is my employee ID?"},
]

show(with_history, "NOW THE FACT IS ACTUALLY IN THE CONTEXT")
print("\nREPLY:", llm(with_history))

NOW THE FACT IS ACTUALLY IN THE CONTEXT — 4 messages, ~69 tokens
[0] system ~6 tok | You are an HR assistant.
[1] user ~16 tok | My employee ID is E-4417 and I work in the Pune office.
[2] assistant ~25 tok | Hello! How can I assist you today with your employee ID E-4417 and your inquiries related to the Pune office?
[3] user ~6 tok | What is my employee ID?

REPLY: Your employee ID is E-4417. If you need any further assistance, feel free to ask!


### Why this one idea explains the rest of the course

Because you resend everything every turn:

- turn 20 costs roughly **20 times** turn 1 → *week 3, context budgets*
- the conversation eventually exceeds the window → *week 3, compression*
- you must choose what is worth resending → *week 3, prioritisation*
- documents cannot all be resent → *weeks 4 and 5, RAG*
- an agent's loop re-sends its whole trajectory → *week 7, context growth*

Every one of those is a consequence of this cell.

---

## Part 7 — The same input can give different output

The model produces a probability distribution and then **samples** from it. Two identical
calls can differ. Treat an LLM call like a flaky network call, never like a pure function.

- `temperature` — flattens or sharpens the distribution. Low is focused, high is varied.
- `top_p` — samples only from the most probable slice. Tune one or the other, not both.

In [36]:
q = [
    {"role": "system", "content": "You are an HR assistant. One short sentence only."},
    {"role": "user",   "content": "Describe our leave policy in your own words."},
]

rows = []
for temp in (0.0, 0.0, 1.3, 1.3):
    rows.append([temp, llm(q, temperature=temp)[:500]])

table(rows, ["temperature", "reply (first 500 chars)"])

+---------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|   temperature | reply (first 100 chars)                                                                                                                                           |
+===============+===================================================================================================================================================================+
|           0   | Our leave policy allows employees to take time off for personal, medical, and vacation needs while ensuring adequate coverage and communication with their teams. |
+---------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|           0   | Our leave policy allows employees to take time off for personal, medical

Two things to take away, and the second one matters more:

1. Low temperature is *more consistent*, *not deterministic*. Never assume repeatability.
2. **If your code depends on the shape of the output, verify it in code.** A prompt that
   says "reply with exactly 5 bullet points" is a request, not a guarantee. Week 2 fixes
   this properly with schemas.

---

## Part 8 — Simple models vs thinking models

Everything in Part 7 — temperature, top-p, sampling — describes one kind of model. There
is a second kind, and **none of Part 7 applies to it.**

| | Simple (sampling) | Thinking (reasoning) |
|---|---|---|
| How it answers | Predicts the next token immediately | Reasons privately first, then answers |
| `temperature` / `top_p` | Apply — you tune them | **Do not apply** |
| What you pay for | Tokens you can see | **Plus thinking tokens you never see** |
| Latency | Fast, predictable | Slower, varies with difficulty |
| How to prompt it | Tell it *how* to think — steps, examples | **Stop telling it how to think.** Give the goal and the constraints |
| Best at | Extraction, classification, routing, chat, high volume | Planning, multi-step logic, hard debugging, ambiguity |
| Cost of choosing wrong | Weak reasoning on a hard task | 10x the money and latency on a trivial one |

Run the same question against both and read the numbers.

In [38]:
question = [
    {"role": "system", "content": "You are an HR assistant."},
    {"role": "user",   "content": (
        "An employee joined on 14 March, has taken 6 days leave, and wants 5 days off "
        "from 22 December. Earned leave accrues at 1.5 days per completed month. "
        "Do they have enough leave? Show your working."
    )},
]

for model in (SIMPLE_MODEL, THINKING_MODEL):
    SPEND.clear()
    answer = llm(question, model=model)
    r = SPEND[-1]
    print(f"--- {model} ---")
    print(f"{r['seconds']:.1f}s | in {r['in']} out {r['out']} | reasoning {r['reasoning']} | ${r['usd']:.6f}")
    print(answer, "\n")

--- gpt-4o-mini ---
4.4s | in 68 out 457 | reasoning 0 | $0.000284
To determine whether the employee has enough earned leave, we need to calculate how many leave days the employee has accrued and whether that covers the 6 days already taken plus the additional 5 days requested.

### Step 1: Calculate the Accrued Leave

1. **Joining Date:** 14 March
2. **Current Date:** Assuming we are calculating this before 22 December, we will consider the end of November as the last complete month prior to the leave request (December).
3. **Complete Months of Service by End of November:**
   - March: 14th to 31st = 0 (not a complete month)
   - April: 1 month
   - May: 1 month
   - June: 1 month
   - July: 1 month
   - August: 1 month
   - September: 1 month
   - October: 1 month
   - November: 1 month
   - **Total Complete Months = 7**

4. **Leave Accrual Rate:** 1.5 days per month

5. **Total Accrued Leave:**
   \( \text{Total Accrued Leave} = 7 \text{ months} \times 1.5 \text{ days/month} = 10.5 

Look at the `reasoning` column for the thinking model. **You paid for those tokens and you
will never see them.** That is the trade: better multi-step answers, in exchange for money
and latency you cannot inspect.

### Does `temperature` even work on a thinking model?

Find out rather than believing a slide.

In [39]:
try:
    out = llm(question, model=THINKING_MODEL, temperature=0.2)
    print("Accepted the parameter. Whether it changed anything is a separate question.")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

BadRequestError: Error code: 400 - {'error': {'message': "Unsupported value: 'temperature' does not support 0.2 with this model. Only the default (1) value is supported.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': 'unsupported_value'}}


### How to choose — the version to remember

- **Routing, extraction, classification, chat, anything high-volume →** simple model.
- **Planning, multi-step arithmetic, ambiguous judgement →** thinking model.
- **Default to the cheap one** and escalate only when you can point at a failure it caused.

We return to this twice: **week 2**, where prompting the two differs more than you expect,
and **week 7**, where a thinking model plans the agent's next step while a cheap model does
the routine work inside the loop.

> **Also worth knowing (and covered in the deck, not the code):** big vs small models,
> closed API vs open-weight, and that model choice is an engineering decision you revisit
> when cost or latency changes — not a brand preference.

---

## Part 9 — `hr-assistant` v0: the conversation loop

Everything so far, assembled. The loop is five lines, and it is the whole of chat:

```mermaid
flowchart LR
    U["user types"] --> AP["append as 'user'"]
    AP --> S["send the WHOLE list"]
    S --> M(("model"))
    M --> AR["append reply as 'assistant'"]
    AR --> U
```

The only class in this notebook. It earns that because it genuinely holds state.

In [41]:
SYSTEM_PROMPT = (
    "You are the HR assistant for a software company in Pune. "
    "Answer employee questions about leave, payroll and workplace policy. "
    "Be concise — 3 sentences at most. "
    "If you do not know something, say so plainly instead of guessing."
)

class Conversation:
    def __init__(self, system=SYSTEM_PROMPT, model=SIMPLE_MODEL):
        self.messages = [{"role": "system", "content": system}]
        self.model = model

    def ask(self, text, verbose=False):
        self.messages.append({"role": "user", "content": text})
        if verbose:
            show(self.messages, f"TURN {sum(1 for m in self.messages if m['role'] == 'user')}")
        reply = llm(self.messages, model=self.model)
        self.messages.append({"role": "assistant", "content": reply})
        return reply

In [42]:
SPEND.clear()
chat = Conversation()

print("Q1:", chat.ask("Hi — my employee ID is E-4417."))
print("\nQ2:", chat.ask("How much notice do I need to give before taking leave?"))
print("\nQ3:", chat.ask("Remind me — what was my employee ID?"))   # it remembers, because YOU resent it

print("\n", spend_summary())

Q1: How can I assist you today regarding leave, payroll, or workplace policy?

Q2: Typically, employees are required to give at least one week's notice for planned leave. However, for unexpected leave, please notify your supervisor as soon as possible. Please refer to the employee handbook for specific details.

Q3: Your employee ID is E-4417.

 3 calls   $0.000089


Run the last turn with `verbose=True` and watch the message array grow. That growth is the
subject of week 3.

---

## Part 10 — Streaming

The model generates one token at a time, so you can display them as they arrive. Nothing
gets faster; it just stops *feeling* slow. Time-to-first-token is what users perceive as
speed, and in week 9 this same idea decides whether a voice agent feels natural or broken.

In [43]:
def ask_streaming(conv, text):
    conv.messages.append({"role": "user", "content": text})
    stream = client.chat.completions.create(
        model=conv.model, messages=conv.messages, stream=True,
        stream_options={"include_usage": True},
    )

    started, first_at, parts, usage = time.time(), None, [], None
    for chunk in stream:
        if chunk.usage:
            usage = chunk.usage
        if not chunk.choices:
            continue
        piece = chunk.choices[0].delta.content
        if piece:
            if first_at is None:
                first_at = time.time() - started
            parts.append(piece)
            print(piece, end="", flush=True)

    reply = "".join(parts)
    conv.messages.append({"role": "assistant", "content": reply})
    if usage:
        SPEND.append({
            "model": conv.model, "in": usage.prompt_tokens, "out": usage.completion_tokens,
            "reasoning": 0, "seconds": time.time() - started,
            "usd": cost_of(conv.model, usage.prompt_tokens, usage.completion_tokens),
        })
    print(f"\n\n[first token after {first_at:.2f}s | complete in {time.time()-started:.2f}s]")
    return reply


ask_streaming(chat, "Can I carry unused leave into next year?")

Unused leave policies may vary, so please refer to the employee handbook for the specific rules regarding carryover of unused leave into the next year. If you have further questions, I recommend checking with your supervisor or HR manager.

[first token after 0.01s | complete in 0.41s]


'Unused leave policies may vary, so please refer to the employee handbook for the specific rules regarding carryover of unused leave into the next year. If you have further questions, I recommend checking with your supervisor or HR manager.'

---

## Part 11 — The cost meter, and the curve that shapes week 3

You resend the whole history every turn, so input tokens grow every turn, so **cost per
turn grows even though the questions stay the same length.**

Measure it rather than being told it.

In [ ]:
SPEND.clear()
demo = Conversation()

questions = [
    "What is the leave policy?", "How do I apply for leave?", "What about sick leave?",
    "Is there a carry-forward limit?", "When does leave lapse?", "Who approves my leave?",
    "Can I take leave in my notice period?", "What about unpaid leave?",
    "How is leave encashed?", "Does leave accrue monthly?",
]
for q in questions:
    demo.ask(q)

rows, running = [], 0.0
for i, r in enumerate(SPEND, start=1):
    running += r["usd"]
    rows.append([i, r["in"], r["out"], f"${r['usd']:.6f}", f"${running:.6f}"])

table(rows, ["turn", "input tok", "output tok", "cost", "running"])

first, last = SPEND[0], SPEND[-1]
print(f"\nTurn 10 sends {last['in']/first['in']:.1f}x the input of turn 1.")
print("Extend to turn 50 and the same conversation costs more than the answers are worth.")
print("That is week 3.")

---

# Lab (50 min)

Work in `lab/`. Two tasks.

### Lab 1 — Break it, then fix it

Take `Conversation` and make one change: in `ask()`, send **only the system prompt and the
current question**, discarding the history.

1. Run a 3-turn conversation where turn 3 depends on turn 1. Watch it fail.
2. Use `show()` to prove *why* it failed — point at the missing message.
3. Restore the history and confirm the fix.

**Acceptance:** you can explain the failure in one sentence without using the word
"forgot". The model did not forget anything; you did not send it.

### Lab 2 — The cost curve

Run a 20-turn conversation and plot input tokens per turn, and cumulative cost per turn.

- What shape is the input-tokens line? Why that shape and not a flat one?
- What would the 50th turn cost?
- Name three things you could drop from the context to flatten it. Which would you drop
  first, and what breaks if you are wrong?

**Acceptance:** a printed table or plot, plus your three candidates ranked. You are
inventing week 3's content an hour before we teach it — that is deliberate.

---

# Homework — mandatory, due 24 hrs before week 2

Submitted as a tagged commit on your own `hr-assistant` repo. `check.py` in the homework
folder tells you pass or fail before you submit.

### 1. Temperature variance table
Same prompt, 10 calls: 5 at `temperature=0`, 5 at `temperature=1.3`. Tabulate the replies.
**Deliver:** the table, plus one sentence on what you would do if your code depended on a
specific output shape.

### 2. Pick a model type, with your own numbers
Run three representative HR questions — one trivial (routing), one factual, one multi-step
— against both models. Record latency, cost and quality.
**Deliver:** which model you are choosing for the HR assistant and why, in five lines,
citing **your** numbers rather than the table in Part 8.

### 3. Add 10 golden questions
A curated set of 30 ships in `corpus/`. Add 10 of your own — real questions an employee
would actually ask, answerable from the corpus, with an expected answer for each.
**These are used from week 5 to week 10 to measure your own system**, so write them as if
someone else has to grade you with them. They will.

### 4. Ship v0
`hr-assistant` v0 running: streaming, multi-turn, cost meter, tagged `v0`.

---

## What you can now explain to someone else

1. An LLM predicts the next token. That is the entire mechanism.
2. Tokens are money and tokens are space.
3. **The model has no memory** — whatever it knows, you put there in this call.
4. Identical calls can differ, so verify in code what you depend on.
5. There are two kinds of model, and temperature only applies to one of them.
6. Cost per turn grows even when your questions do not.

### The question to carry into week 2

Today the assistant answered a leave question **without ever seeing the leave policy** —
confidently, and possibly wrongly. Weeks 2 through 6 are one long answer to that.

Next week: making the output something your code can trust.